In [1]:
# --- 1. Cargar lo que ya tienes ---
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

grid_features = pd.read_csv('../data/processed/grid_features.csv')

COUNTRIES_GEOJSON = "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_50m_admin_0_scale_rank.geojson"
paises = gpd.read_file(COUNTRIES_GEOJSON)

print(paises.columns.tolist())  # confirmar el nombre real de la columna de código de país

['scalerank', 'labelrank', 'sr_sov_a3', 'sr_adm0_a3', 'sr_gu_a3', 'sr_su_a3', 'sr_subunit', 'featureclass', 'geometry']


In [3]:
# --- 2. Convertir el grid a puntos geoespaciales ---
grid_gdf = gpd.GeoDataFrame(
    grid_features,
    geometry=[Point(xy) for xy in zip(grid_features['lon'], grid_features['lat'])],
    crs="EPSG:4326"
)

# --- 3. Spatial join: ¿dentro de qué país cae cada celda? ---
grid_con_pais = gpd.sjoin(
    grid_gdf,
    paises[['sr_adm0_a3', 'geometry']].rename(columns={'sr_adm0_a3': 'iso_a3'}),
    how='left',
    predicate='within'
)

# --- 4. Comprobación de sanidad, antes de exportar ---
print(f"Celdas totales: {len(grid_con_pais)}")
print(f"Celdas SIN país asignado (esperado: mar abierto): {grid_con_pais['iso_a3'].isna().sum()}")
print(grid_con_pais[grid_con_pais['iso_a3'].isna()][['lat', 'lon']].head())

# --- 5. Exportar ---
grid_con_pais.drop(columns='geometry').to_csv(
    '../data/processed/grid_features_con_pais.csv', index=False
)

Celdas totales: 41162
Celdas SIN país asignado (esperado: mar abierto): 29562
         lat         lon
0 -19.153525   50.159594
1  21.114410 -167.893271
2  -0.491605   55.368580
3 -50.198830   23.374294
4  69.434243   45.469661


In [5]:
# --- 6. PIB per cápita por país (Banco Mundial, API pública y gratis) ---
import requests

def obtener_indicador_banco_mundial(indicador: str, anio: str = "2023") -> pd.DataFrame:
    """Descarga un indicador del Banco Mundial para todos los países,
    para el año más reciente disponible."""
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicador}"
    params = {"format": "json", "per_page": 300, "date": anio}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    registros = data[1]  # data[0] es metadata de paginación, data[1] son los datos reales
    filas = [
        {"iso_a3": r["countryiso3code"], "valor": r["value"]}
        for r in registros if r["value"] is not None
    ]
    return pd.DataFrame(filas)

pib = obtener_indicador_banco_mundial("NY.GDP.PCAP.CD").rename(columns={"valor": "pib_per_capita"})
poblacion = obtener_indicador_banco_mundial("SP.POP.TOTL").rename(columns={"valor": "poblacion"})

print(f"PIB: {len(pib)} países | Población: {len(poblacion)} países")
print(pib.head())

PIB: 251 países | Población: 264 países
  iso_a3  pib_per_capita
0    AFE     1571.132704
1    AFW     1846.246811
2    ARB     7504.356377
3    CSS    17609.682707
4    CEB    22693.950447


In [8]:
# --- 7. Filtrar agregados regionales del Banco Mundial (no son países) ---
resp_meta = requests.get(
    "https://api.worldbank.org/v2/country",
    params={"format": "json", "per_page": 400}
)
paises_meta = resp_meta.json()[1]

meta = pd.DataFrame([
    {"iso_a3": p["id"], "nombre": p["name"], "region": p["region"]["value"]}
    for p in paises_meta
])

codigos_agregados = meta.loc[meta["region"] == "Aggregates", "iso_a3"]

print(f"Agregados a excluir: {len(codigos_agregados)}")

pib = pib[pib["iso_a3"].notna() & (pib["iso_a3"].str.len() == 3)]
poblacion = poblacion[poblacion["iso_a3"].notna() & (poblacion["iso_a3"].str.len() == 3)]
pib = pib[~pib["iso_a3"].isin(codigos_agregados)]
poblacion = poblacion[~poblacion["iso_a3"].isin(codigos_agregados)]

print(f"PIB (solo países reales): {len(pib)} | Población (solo países reales): {len(poblacion)}")
print(pib.head())

Agregados a excluir: 78
PIB (solo países reales): 204 | Población (solo países reales): 217
   iso_a3  pib_per_capita
47    AFG      413.757895
48    ALB     9740.702341
49    DZA     5370.477235
50    AND    46812.448450
51    AGO     2885.513491


In [9]:
# --- 8. Unir todo: celda -> país -> PIB/población ---
grid_enriquecido = (
    grid_con_pais
    .merge(pib, on="iso_a3", how="left")
    .merge(poblacion, on="iso_a3", how="left")
)

print("Celdas sin PIB (aparte de las de océano, esperado):")
print(grid_enriquecido[grid_enriquecido["iso_a3"].notna() & grid_enriquecido["pib_per_capita"].isna()]["iso_a3"].unique())

grid_enriquecido.drop(columns="geometry", errors="ignore").to_csv(
    "../data/processed/grid_features_enriquecido.csv", index=False
)

Celdas sin PIB (aparte de las de océano, esperado):
['ATA' 'YEM' 'CUB' 'SDS' 'SAH' 'KOS' 'PRK' 'ERI' 'SOL' 'SYR' 'TWN' 'ATF'
 'FLK']
